### RAG for quality Genie;

- Genie Space: RAG tailored spaces;
- Table metadata, table tags, table description;
- Data model relationships beyond Primary Key & Foreign Keys, metadata for Semantic relationship;
- Sql-queries for cached context;

### Create vector search endpoints and indexes

#### ai_parse_document

In [0]:
%sql
CREATE OR REPLACE TABLE llm.rag.ai_parsed_documents
USING DELTA
AS
SELECT
    ai_parse_document(content) AS parsed_content

FROM READ_FILES('/Volumes/llm/ai_parse/ai_parse_documents', format => 'binaryFile');

#### ai_extract concatanated with ai_parse_document

In [0]:
%sql
SELECT
  extracted_variant:response.company_name.value::STRING AS company_name,
  extracted_variant:response.price.value::STRING AS price,
  extracted_variant:response.reporting_period.value::STRING AS reporting_period
FROM (
  SELECT
    ai_extract(
      ai_parse_document(content),
      '["company_name", "reporting_period", "price"]'
    ) AS extracted_variant
  FROM READ_FILES('/Volumes/llm/ai_parse/ai_parse_documents', format => 'binaryFile')
) extracted_docs;

#### Creating a Vector Search Endpoint

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient

In [0]:
client = VectorSearchClient()
# client = VectorSearchClient(service_principal_client_id=<CLIENT_ID>,service_principal_client_secret=<CLIENT_SECRET>)
client.create_endpoint(
    name="vector_search_ai_parsed_documents",
    endpoint_type="STANDARD" # or "STORAGE_OPTIMIZED"
)

#### Updating vector search endpoint

In [0]:
# from databricks.vector_search.client import VectorSearchClient, MIN_QPS_RESET_TO_DEFAULT

# client = VectorSearchClient()

# # Set or update minimum QPS
# response = client.update_endpoint(name="vector_search_endpoint_name", min_qps=500)

# # Check scaling status
# scaling_info = response.get("endpoint", {}).get("scaling_info", {})
# print(f"State: {scaling_info.get('state')}")  # SCALING_CHANGE_IN_PROGRESS or SCALING_CHANGE_APPLIED

# # Remove high QPS configuration and return to default
# client.update_endpoint(name="vector_search_endpoint_name", min_qps=MIN_QPS_RESET_TO_DEFAULT)

#### Extracting dataset from 5 different .pdfs

In [0]:
%sql
CREATE OR REPLACE TABLE llm.rag.ai_parsed_documents_extracted
SELECT
  extracted_variant:response.company_name.value::STRING AS company_name,
  extracted_variant:response.price.value::STRING AS price,
  extracted_variant:response.reporting_period.value::STRING AS reporting_period,
  extracted_variant:response.history.value::STRING AS history
FROM (
  SELECT
    ai_extract(
      ai_parse_document(content),
      '["company_name", "reporting_period", "price", "history"]'
    ) AS extracted_variant
  FROM READ_FILES('/Volumes/llm/ai_parse/ai_parse_documents', format => 'binaryFile')
) extracted_docs;

In [0]:
%sql
ALTER TABLE llm.rag.ai_parsed_documents_extracted SET TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [0]:
%sql
ALTER TABLE llm.rag.ai_parsed_documents_extracted SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 7 days')

#### Creating Delta synch to index;

- Using extracted dataset to build an index inside the vector search store; future use -> similarity search;


In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

index = vsc.create_delta_sync_index(
    endpoint_name="vector_search_ai_parsed_documents",
    source_table_name="llm.rag.ai_parsed_documents_extracted",
    index_name="llm.rag.company_index",
    pipeline_type="TRIGGERED",
    primary_key="company_name",
    embedding_source_column="history",
    embedding_model_endpoint_name="databricks-gte-large-en"
)
index.describe()['status']['message']

#### Updating source table

In [0]:
from pyspark.sql.functions import lit

target_table = "llm.rag.ai_parsed_documents_extracted"

# Upsert fake row using MERGE
spark.sql(f"""
MERGE INTO {target_table} AS target
USING (SELECT 'RealREalCorp' AS company_name, '54321' AS price, '2027' AS reporting_period, 'RealREalCorp was founded in 2027 specializes in AI solutions to make your life wonderful.' AS history) AS source
ON target.company_name = source.company_name
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

- checking table

In [0]:
%sql
SELECT * FROM llm.rag.ai_parsed_documents_extracted

- Checking index to see if it was updated based on upsert in table; automatic sync between source table and vector index;

In [0]:
index = vsc.create_delta_sync_index(
    endpoint_name="vector_search_ai_parsed_documents",
    source_table_name="llm.rag.ai_parsed_documents_extracted",
    index_name="llm.rag.company_index",
    pipeline_type="TRIGGERED",
    primary_key="company_name",
    embedding_source_column="history",
    embedding_model_endpoint_name="databricks-gte-large-en"
)

- Querying index for similarity search

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
index = client.get_index(endpoint_name="vector_search_ai_parsed_documents", index_name="llm.rag.company_index")

In [0]:
# Perform a semantic similarity search and specify columns to return
results = index.similarity_search(
    query_text="Show me which company reported in 2026-Q1",
    num_results=1,
    query_type="ann",  # Approximate Nearest Neighbor (Default)
    columns=["company_name", "price", "reporting_period", "history"]
)

display(results)


#### Re-ranker

In [0]:
from databricks.vector_search.reranker import DatabricksReranker

# Perform a semantic similarity search with a reranker and specify columns to return
results = index.similarity_search(
    query_text="Show me which company reported in 2026-Q1",
    num_results=1,
    query_type="ann",  # Approximate Nearest Neighbor (Default)
    columns=["company_name", "price", "reporting_period", "history"],
    reranker=DatabricksReranker(columns_to_rerank=["reporting_period"])
)

display(results)